# Optimize an OpenTools agent with DSPy

This notebook follows DSPy's advanced tool-use pattern: OpenTools supplies policy-gated callables, a DSPy `Module` implements the tool-selection trajectory, and SIMBA optimizes that agent policy from real examples and a metric. It does **not** optimize the tool implementation itself.

Install with `python -m pip install -e '.[dspy]'`. Provider-backed baseline evaluation and optimization are opt-in because they make real model calls and may incur cost.

In [ ]:
from pathlib import Path
import os
import sys

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/opentools').is_dir()), None)
if REPO is None:
    raise RuntimeError('Run this notebook from within the OpenTools repository.')
sys.path.insert(0, str(REPO / 'src'))

import dspy
from opentools.integrations.dspy import build_dspy_agent, optimize_dspy_agent
print('DSPy version:', dspy.__version__)


## 1. Build the compilable agent

The agent contains a DSPy `ChainOfThought` policy. Calculator and `finish` are real Python callables described to that policy.

In [ ]:
agent = build_dspy_agent(['Calculator_Tool'], max_steps=4)
print(agent)
print('Available functions:', sorted(agent.functions))
print('Calculator metadata:', agent.tools['Calculator_Tool'])

# This direct call proves the underlying OpenTools function is real; it makes no LM call.
observed = agent.functions['Calculator_Tool'](operation='multiply', values=[6, 7])
print('Direct observed result:', observed)
assert observed['result'] == '42'


## 2. Define real optimization and evaluation examples

Replace these small examples with representative tasks from your target agent workload. Keep a separate development set so the optimizer is not evaluated only on its training examples.

In [ ]:
def example(question, answer):
    return dspy.Example(question=question, answer=answer).with_inputs('question')

trainset = [
    example('What is 2 plus 3?', '5'),
    example('What is 6 multiplied by 7?', '42'),
    example('What is the square root of 81?', '9'),
    example('What is 100 divided by 4?', '25'),
]
devset = [
    example('What is 14 plus 9?', '23'),
    example('What is 8 multiplied by 12?', '96'),
]

def normalized_exact_match(example, prediction, trace=None):
    gold = str(example.answer).strip().lower().replace(',', '')
    predicted = str(prediction.answer).strip().lower().replace(',', '')
    return predicted == gold

print(f'Train examples: {len(trainset)}; dev examples: {len(devset)}')


## 3. Configure a real model and measure the baseline

Set `RUN_DSPY_OPTIMIZATION=1`, configure the provider credential required by `DSPY_LM`, and rerun this cell. No baseline score is generated when model execution is disabled.

In [ ]:
RUN_OPTIMIZATION = os.getenv('RUN_DSPY_OPTIMIZATION') == '1'

if not RUN_OPTIMIZATION:
    print('NOT RUN: set RUN_DSPY_OPTIMIZATION=1 and real provider credentials.')
else:
    lm_name = os.getenv('DSPY_LM', 'openai/gpt-4o-mini')
    lm = dspy.LM(lm_name, temperature=0.2)
    dspy.configure(lm=lm)
    evaluate = dspy.Evaluate(
        devset=devset, metric=normalized_exact_match,
        num_threads=1, display_progress=True, display_table=0,
    )
    baseline_score = evaluate(agent)
    print('Observed baseline score:', baseline_score)


## 4. Compile the agent policy with SIMBA

SIMBA can revise the DSPy policy instructions and add demonstrations based on the metric. `batch_size` must not exceed the trainset size. This small configuration is for verifying the path; use a larger representative trainset for paper experiments.

In [ ]:
if not RUN_OPTIMIZATION:
    print('NOT RUN: no optimized score is claimed.')
else:
    optimized_agent = optimize_dspy_agent(
        agent,
        trainset=trainset,
        metric=normalized_exact_match,
        batch_size=4,
        max_steps=2,
        max_demos=2,
        seed=0,
        num_threads=1,
    )
    optimized_score = evaluate(optimized_agent)
    print('Observed optimized score:', optimized_score)
    print('Observed score change:', optimized_score - baseline_score)
    optimized_agent.save('opentools_dspy_agent.json')
    print('Saved compiled agent to opentools_dspy_agent.json')


## Interpretation

`as_dspy_tool(...)` is only an interoperability adapter. `build_dspy_agent(...)` creates the DSPy program that decides which OpenTools function to call, with which arguments, and when to finish. `optimize_dspy_agent(...)` compiles that program against your examples and metric. Report baseline and optimized scores only after the provider-backed cells actually run, and evaluate on held-out tasks.